# ATLOP Baseline 재현 (Revised 데이터, Colab A100)

> **최종 업데이트**: 2026-07-15 (**PUATLoss(0.7) 실험 결과 기록 — baseline 대비 개선 없음
> (예상대로); Entity-Pair Graph만 단독 적용한 dk_gat 실험으로 교체**):
>
> **1) PUATLoss(0.7) 실행 결과 확정** (`run_name=atlop_pu07_revised`, 셀 5의 실제 로그 기준):
> `--early_stop_patience 5`가 실제로 발동해 **epoch 13에서 조기 종료**(best epoch 8, 8 이후
> 5 epoch 연속 미개선). best 체크포인트 기준 dev F1 **73.30**/Ign F1 **72.20**, test(best
> ckpt) F1 **73.67**/Ign F1 **72.60**(P 81.56/R 67.18). plain 베이스라인(dev F1 73.67/Ign
> 72.95, 실제 peak epoch14 73.97/73.14)과 비교하면 **dev 기준으로 오히려 −0.37~−0.94 낮음**
> — 바로 아래 이전 업데이트에서 예상했던 대로("이 데이터셋의 노이즈는 positive 쪽 과다추정이라
> na_weight=0.7이 겨냥하는 Na쪽 노이즈와 메커니즘이 다름") **개선되지 않음이 실측으로
> 확인됨**. baseline을 대체하지 않고 "결과 기록" 표에 실험 결과로만 남김 — PU loss는 이
> 데이터셋에서 더 이상 시도하지 않음.
>
> **2) 다음 실행: Entity-Pair Graph만 단독 적용** (`run_name=dk_gat_pairgraph_revised`,
> 셀 5 교체): 모델을 `Scripts.dk_gat.train_baseline`(그래프 없음)에서
> `Scripts.dk_gat.train_gat`(GAT, `Scripts/dk_gat/model.py`)로 바꾸되, 과거에 이미 안 좋았던
> 요소는 전부 빼고 **Entity-Pair Graph(`--use_pair_graph`) 하나만** 추가함:
> - **빼는 것**: `--use_gated_fusion`/`--use_bilinear_classifier` — 원본 데이터에서 distant
>   스크리닝은 각각 +2.2/+3.67 F1로 좋았지만, 실제 15epoch 풀런에서 baseline(61.71) 대비
>   오히려 60.62로 낮게 나온 조합(`results/comparison.md`, [[project-dk-gat-status]]). PU
>   loss도 위 1번 결과로 제외.
> - **넣는 것**: `--use_pair_graph --pair_graph_dim 256` — zero-init 잔차 헤드라 켜도 최소
>   init 시점엔 안 켠 것과 동일(해가 될 여지가 작음), 노리는 게 "A→B,B→C⇒A→C 합성 추론"인데
>   `relations_revised.csv` 확인상 `train_revised.json` 라벨의 28.8%가 정확히 이런 합성
>   추론(`unresolved_multihop`, 근거 문장 없음)이라 이 데이터셋과 특히 맞아떨어질 가능성이
>   있음(2턴 전 대화 참고). 단, 원본 데이터에서도 단독 검증된 적은 없음(항상 다른 미검증
>   요소와 번들로만 시도됨) — 확실한 개선이라 단정할 근거는 없고, 이번이 사실상 첫 단독
>   실측.
> - JK(Jump Knowledge)는 기본값 유지(off로 끄지 않음, Gated Fusion 안 켜므로 JK가 결합
>   단계를 담당), Meta-path Attention/Curriculum PU-weight/layerwise_lr_decay/
>   freeze_encoder_epochs/evidence_start_epoch/evidence_fusion도 전부 미적용(기본값) —
>   pair-graph 하나의 효과만 baseline과 비교하기 위해 나머지는 전부 원래 GAT 구조(Entity+
>   Sentence heterogeneous graph + JK + interaction pair repr, 확정 파이프라인 그대로) 그대로 둠.
> - `--early_stop_patience 5` + best-epoch 체크포인트는 `train_gat.py`에 이미 있는 기능 그대로
>   사용. 스크리닝 없이 바로 전체 20 epoch 실행(사용자 결정) — CPU smoke test
>   (`--limit_docs 6 --epochs 3 --use_pair_graph --early_stop_patience 2`)로 크래시 없음만
>   확인, 성능 사전 확인은 생략.
>
> 이전 (**PUATLoss(na_weight=0.7)를 revised 데이터셋에 실험 적용 + early stopping(patience=5) +
> best-epoch 체크포인트 저장 로직 추가; 방금 완료된 순수 baseline 실행을 revised 데이터셋의
> baseline으로 확정**):
>
> **1) 방금 완료된 실행 결과 기록** (`run_name=atlop_baseline_revised`, PUATLoss/early
> stopping/best-checkpoint 전부 없는 순수 baseline, 셀 5의 실제 로그 기준): 마지막(20번째,
> epoch 19) dev F1 73.67/Ign F1 72.95, test F1 73.48/Ign F1 72.77 (P 87.54/R 63.31). **peak는
> epoch 14였음**(dev F1 73.97/Ign F1 73.14) — 당시엔 best-checkpoint 저장 로직이 없어 그
> 체크포인트 자체는 저장되지 못하고 마지막 epoch(19) 것만 남음, 최종 성적과 차이는
> 미미(−0.30/−0.19)하지만 정확히 이 문제(비단조 dev F1 때문에 최고점을 놓침)를 막기 위해
> 이번 업데이트로 best-checkpoint 저장을 추가함. 이 수치를 **revised 데이터셋의
> (plain ATLoss) baseline**으로 맨 아래 "결과 기록" 표에 기입함 — `dev_revised.json`/
> `test_revised.json`(각 500문서) 기준이라 원본 61.71/59.86과 직접 비교 불가하다는 건
> 기존과 동일.
>
> **2) PUATLoss(na_weight=0.7) 실험 적용 — ⚠️ 이 데이터셋에서 검증된 값은 아님**: 원본
> 데이터에서는 `ATLOP + PUATLoss(0.7)`이 검증됐지만(트랙3 `atlop_full_pu07`, dev F1
> 62.06/Ign F1 60.16 — plain ATLoss 대비 +0.35/+0.30, `results/comparison.md`), **revised
> 데이터셋에도 그대로 통할 거라는 근거는 없음**을 확인함 — `docred_data/data/
> relations_revised.csv`를 직접 까본 결과, `train_revised.json`의 relation label 중
> `evidence_source=annotated`(사람 검증)는 53.0%뿐이고 나머지는 `unresolved_multihop`
> (28.8%, 근거 문장 없이 자동 추론된 다중홉 합성) + `inferred_cooccurrence`(18.2%, 공출현
> 휴리스틱 추론)임(`confidence` 컬럼은 전부 1.0이라 신뢰도 구분에 못 씀). 즉 이 데이터셋의
> 잠재적 노이즈는 **positive label 쪽 과다추정**(false positive 위험)이지, na_weight=0.7이
> 원래 겨냥하는 **distant supervision의 Na(negative) 쪽 노이즈**와 메커니즘이 다름 — 위
> 최상단 1번에서 이 우려가 실측으로 확인됨.
>
> **3) early stopping(patience=5) + best-epoch 체크포인트 저장**: `train_gat.py`에 이미 있던
> 것과 동일한 패턴을 `train_baseline.py`에도 포팅 — "마지막 epoch이 최고가 아닐 수 있다"는
> 문제를 방지(데이터셋과 무관하게 항상 유효, 위 최상단 2번의 pair-graph 실행에도 그대로 적용).
>
> 이전 (**dk EGAT 실험 중단, ATLOP baseline 재현 파이프라인으로 원복**):
> `dk_gat` 트랙(그래프 증강)에서 revised 데이터로 재학습을 돌리다가, **원래 검증된 baseline
> 아키텍처**(BERT-base-cased + Sliding Window + Entity Marker/logsumexp Pooling + Localized
> Context Pooling + `[Entity;Context]→Linear→Tanh(1536→768)` + Grouped Bilinear + ATLoss +
> Adaptive Thresholding, 원본 데이터 기준 dev F1 61.71/Ign F1 59.86 검증됨)로 되돌리기로 함 —
> GAT/그래프 관련 구성 요소(Gated Fusion, Meta-path Attention, Entity-Pair Graph, Evidence
> Contrastive Loss 등)는 전부 제거. **데이터셋은 바로 이전과 동일하게 유지** —
> `train_revised.json`(distant 없음, 최대 20 epoch), `dev_revised.json`으로 매 epoch 검증,
> `test_revised.json`으로 최종 트리플 예측.
>
> 이 baseline은 `Scripts/atlop/re_model.py`/`preprocess.py`와 **정확히 동일한 수식**으로
> 재구현했지만(`Scripts/dk_gat/model_baseline.py`/`preprocess_baseline.py`), `Scripts/atlop`는
> 팀원 트랙이라 그 파일들을 직접 수정/import하지 않고 `Scripts/dk_gat/` 안에 완전히 자체
> 구현함 — 상세는 `Scripts/dk_gat/README.md`의 관할 구분 참고.
>
> 참고로 바로 이전 GAT 실행(`dk_gat_revised`, 이 노트북에서 중단됨)의 로그는 epoch 3까지
> dev F1 61.50/Ign F1 60.71까지 순조롭게 올라가고 있었음 — encoder unfreeze(epoch 1) 이후
> 정상 궤도였던 것으로 보임. 이번 baseline 결과와 비교해볼 수 있도록 남겨둠.
>
> ⚠️ **주의**: `dev_revised.json`/`test_revised.json`은 각각 500문서로, 원본 baseline
> 61.71/59.86은 원본 `dev.json`(998문서) 기준이라 **직접 비교 불가** — 이번 실행 결과는 새
> 표로 따로 기록할 것 (맨 아래 "결과 기록" 셀 참고).

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: `train_revised.json`(3,053개, distant 없음) × 최대 **20 epoch**, dk EGAT
모델(BERT-base-cased + Entity Marker/logsumexp Pooling + Localized Context Pooling +
Entity+Sentence Heterogeneous GAT + JK + interaction pair repr) + **Entity-Pair Graph
단독 추가**(`--use_pair_graph`, A→B,B→C⇒A→C 합성 추론 타겟) + plain ATLoss(PU loss 미적용,
위 참고) + 0.2×Evidence Contrastive Loss, dev F1이 **5 epoch** 연속 개선 없으면 조기 종료
(`--early_stop_patience 5`), `dev_revised.json`으로 매 epoch 검증 + best-epoch 체크포인트
갱신, `test_revised.json`으로 (best-checkpoint 기준) 최종 트리플 예측 + F1/Ign F1 산출.


In [9]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-e517e286-953a-2010-04a8-00f7d5898935)


In [10]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
#    distant supervision을 안 쓰므로 train_distant.json(417MB)은 pull하지 않음
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/train_revised.json,docred_data/data/dev_revised.json,docred_data/data/test_revised.json,docred_data/data/rel_info.json"
# json들이 KB~MB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*revised* docred_data/data/rel_info.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

Cloning into 'Information_Extraction'...
remote: Enumerating objects: 567, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 567 (delta 37), reused 40 (delta 30), pack-reused 498 (from 1)
Receiving objects: 100% (567/567), 44.64 MiB | 11.56 MiB/s, done.
Resolving deltas: 100% (311/311), done.
Updating files: 100% (166/166), done.
/content/Information_Extraction/Information_Extraction/Information_Extraction
Branch 'dk' set up to track remote branch 'dk' from 'origin'.
Switched to a new branch 'dk'
-rw-r--r-- 1 root root 3.1M Jul 15 02:47 docred_data/data/dev_revised.json
-rw-r--r-- 1 root root 2.7M Jul 15 02:47 docred_data/data/entities_revised.csv
-rw-r--r-- 1 root root  69M Jul 15 02:47 docred_data/data/pinecone_upsert_revised.jsonl
-rw-r--r-- 1 root root  31M Jul 15 02:47 docred_data/data/relations_revised.csv
-rw-r--r-- 1 root root 2.4K Jul 15 02:47 docred_data/data/rel_info.json
-rw-r--r-- 1 root root  906 Jul 15 02:4

In [11]:
# 2) 패키지 (모델은 허브가 아니라 로컬 파일에서 로드할 거라 aria2도 여기서 같이 설치)
!pip install -q transformers==4.57.6 accelerate
!apt-get -qq install -y aria2 > /dev/null

In [12]:
%%bash
# 3) bert-base-cased를 로컬로 직접 받기 (허브 다운로더 대신 aria2c 사용)
#    Colab <-> HF CDN 네트워크가 불안정해서 순정 다운로더는 느리게라도 버텼지만(최저 158kB/s),
#    hf_transfer는 세그먼트 하나가 끊기면 재시도 없이 그대로 멈춰버리는 것으로 확인됨
#    (65MB/s로 빠르게 가다 15%에서 하드행). aria2c는 -x/-s로 멀티커넥션, 끊기면 그 세그먼트만
#    재시도(--max-tries, --retry-wait)하므로 이 네트워크 환경에서 훨씬 안정적.
#    exit 22(HTTP response header bad)를 겪은 적 있어서 세 가지 보강:
#    (a) Colab은 세션마다 /content가 초기화되므로 --continue로 이어받다 손상된 부분파일과
#        충돌하는 걸 막기 위해 매번 디렉토리를 비우고 새로 받음.
#    (b) 커넥션 수를 16->8로 낮춰 HF CDN 레이트리밋/커넥션 리셋 가능성을 줄임.
#    (c) set -e만으로는 "일부 파일이 안 받아진 채로 학습 셀이 그대로 실행되는" 사고를
#        못 막으므로(루프 중간에 실패하면 그 시점까지 받은 파일만 남음), 루프 뒤에
#        파일별 최소 용량 검증을 추가해 하나라도 모자라면 이 셀 자체를 실패시킴.
set -e
rm -rf /content/bert-base-cased
mkdir -p /content/bert-base-cased
BASE_URL="https://huggingface.co/bert-base-cased/resolve/main"
for fname in model.safetensors config.json vocab.txt tokenizer_config.json tokenizer.json; do
  aria2c -x 8 -s 8 -k 1M --max-tries=20 --retry-wait=3 --continue=false \
    -d /content/bert-base-cased -o "$fname" \
    "$BASE_URL/$fname"
done

echo "--- 받은 파일 확인 ---"
ls -lh /content/bert-base-cased

python3 - <<'PY'
import os, sys
min_size = {
    "model.safetensors": 400_000_000,
    "config.json": 200,
    "vocab.txt": 100_000,
    "tokenizer_config.json": 10,
    "tokenizer.json": 300_000,
}
base = "/content/bert-base-cased"
bad = []
for fname, min_bytes in min_size.items():
    path = os.path.join(base, fname)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    if size < min_bytes:
        bad.append(f"{fname}: {size}B (기대 최소 {min_bytes}B)")
if bad:
    print("다운로드 불완전 - 아래 파일이 부족합니다. 이 셀을 다시 실행하세요:", file=sys.stderr)
    for line in bad:
        print(" -", line, file=sys.stderr)
    sys.exit(1)
print("모든 파일 크기 정상 확인 완료.")
PY


07/15 02:47:49 [NOTICE] Downloading 1 item(s)

07/15 02:47:49 [NOTICE] CUID#7 - Redirecting to https://cas-bridge.xethub.hf.co/xet-bridge-us/621ffdc036468d709f174331/83c31be240458b001866527feebc3cece210a4aec957064b2f166d2dd6e8471f?Expires=1784087269&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYyMWZmZGMwMzY0NjhkNzA5ZjE3NDMzMS84M2MzMWJlMjQwNDU4YjAwMTg2NjUyN2ZlZWJjM2NlY2UyMTBhNGFlYzk1NzA2NGIyZjE2NmQyZGQ2ZTg0NzFmKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDA4NzI2OX19fV19&Signature=MEUCIBXMlUD32pRFRIIzJOp4JeVfvuSWVzVCiOaAp9ZIHZ29AiEAy4DDHwPaMGRCiBsB0QnhA55soXEVDcJFidGTZZDlpr8_&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260715%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260715T024749Z&X-Amz-Expires=3600&X-Amz-SignedHeaders=h

In [ ]:
# 4) 학습: dk EGAT + Entity-Pair Graph 단독 적용 (PU loss/Gated Fusion/Bilinear Classifier/
#    Meta-path Attention 등 다른 요소는 전부 미적용), train_revised.json만 사용 (distant 없음)
#    Scripts.dk_gat.train_baseline(그래프 없음) 대신 Scripts.dk_gat.train_gat 사용 --
#    Entity+Sentence Heterogeneous GAT + JK + interaction pair repr(확정 파이프라인)에
#    --use_pair_graph만 추가.
#    --use_pair_graph --pair_graph_dim 256: A->B,B->C=>A->C 합성 추론 타겟, zero-init
#    잔차 헤드라 안 켠 것보다 나빠질 위험은 작음. train_revised.json 라벨의 28.8%가
#    unresolved_multihop(근거 문장 없는 합성 추론)이라 이 데이터셋에 특히 맞을 가능성이
#    있어 시도 -- 단 원본 데이터에서도 단독 검증된 적은 없어 이번이 사실상 첫 실측(스크리닝
#    생략, 바로 20epoch).
#    아래는 의도적으로 뺀 것들: --use_gated_fusion/--use_bilinear_classifier(원본 데이터
#    15epoch 풀런에서 baseline 61.71 대비 60.62로 오히려 낮게 나온 조합), --use_pu_loss
#    (바로 위 atlop_pu07_revised 실행에서 이 데이터셋 dev F1이 baseline보다 낮게 나옴 --
#    "결과 기록" 셀 참고), --use_metapath_attention/--curriculum_na_weight/
#    --layerwise_lr_decay/--freeze_encoder_epochs/--evidence_start_epoch/--evidence_fusion
#    (전부 미검증, 이번엔 pair-graph 하나의 효과만 보기 위해 기본값 유지).
#    --early_stop_patience 5 + best-epoch 체크포인트는 train_gat.py에 이미 있는 기능.
!python -m Scripts.dk_gat.train_gat \
  --model_name_or_path /content/bert-base-cased \
  --train_split docred_data/data/train_revised.json \
  --dev_split docred_data/data/dev_revised.json \
  --test_file docred_data/data/test_revised.json \
  --distant_epochs 0 --epochs 20 \
  --use_pair_graph --pair_graph_dim 256 \
  --early_stop_patience 5 \
  --run_name dk_gat_pairgraph_revised --save_model --seed 66

In [ ]:
# 5) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
#    best-epoch 체크포인트/예측({run_name}_best.pt, {run_name}_best_dev_predictions.json)도
#    마지막 epoch 것과 함께 백업.
#    참고: 직전 atlop_pu07_revised 실행 결과(results/atlop_pu07_revised*)는 이미 "결과 기록"
#    셀에 수치로 옮겨뒀고 baseline보다 낮아 채택하지 않으므로, 용량 절약을 위해 이번엔
#    새 run_name(dk_gat_pairgraph_revised) 파일만 백업함 -- 필요하면 atlop_pu07_revised*도
#    같은 방식으로 별도 백업 가능.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat_pairgraph_revised.pt \
   results/dk_gat_pairgraph_revised_best.pt \
   results/dk_gat_pairgraph_revised_dev_predictions.json \
   results/dk_gat_pairgraph_revised_best_dev_predictions.json \
   results/dk_gat_pairgraph_revised_test_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat_pairgraph_revised

## 결과 기록

**주의**: 이 실행들은 `dev_revised.json`/`test_revised.json`(각 500문서) 기준이라 원본 baseline
61.71/59.86(원본 `dev.json`, 998문서 기준)과 직접 비교할 수 없음 -- 아래처럼 새 표로 따로
관리함.

| 모델 | epochs | dev F1 | dev Ign F1 | test F1 | test Ign F1 |
|---|---|---|---|---|---|
| **ATLOP baseline 재현 (revised, `run_name=atlop_baseline_revised`)** — revised 데이터셋 baseline | 20 (early stop 없음, 마지막 epoch 기준) | 73.67 | 72.95 | 73.48 | 72.77 |
| ATLOP + PUATLoss(0.7) + early stop(5) + best-epoch (revised, `run_name=atlop_pu07_revised`) — **채택 안 함, baseline보다 낮음** | 14 (early stop epoch13, best epoch8 기준) | 73.30 | 72.20 | 73.67 | 72.60 |
| dk EGAT + Entity-Pair Graph 단독 + early stop(5) + best-epoch (revised, `run_name=dk_gat_pairgraph_revised`) | (실행 후 기입) | | | | |

- seed 66 단일 시드.
- baseline 행은 2026-07-15 Colab A100 실측(셀 5 로그 기준) — dev F1이 마지막 epoch(19)까지
  단조 증가하지 않았고 **실제 peak는 epoch 14(dev F1 73.97 / Ign F1 73.14)** 였음(최종 대비
  −0.30/−0.19, 미미하지만 이 문제를 막기 위해 이후 실행부터 best-epoch 체크포인트 저장을
  추가함 -- 맨 위 "최종 업데이트" 참고). 당시엔 저장 로직이 없어 그 체크포인트 자체는
  남지 않음.
- **PUATLoss 행**: `--early_stop_patience 5`가 실제로 발동해 epoch 13에서 조기 종료(best는
  epoch 8). best 체크포인트 기준 dev F1/Ign F1이 baseline(final 73.67/72.95, peak
  73.97/73.14)보다 **낮게** 나옴(−0.37~−0.94) — na_weight=0.7은 원본 데이터의 distant
  Na(negative) 라벨 노이즈를 겨냥한 값인데, revised 데이터의 잠재 노이즈는 positive label
  과다추정(`relations_revised.csv` 기준 라벨의 47%가 자동 추론) 쪽이라 메커니즘이 안 맞았던
  것으로 추정(맨 위 "최종 업데이트" 참고). **이 데이터셋에서 PU loss는 더 이상 시도하지
  않음.**
- **Entity-Pair Graph 행**: 과거 원본 데이터에서 낮게 나왔던 Gated Fusion/Bilinear
  Classifier는 빼고, Entity-Pair Graph(`--use_pair_graph`)만 단독으로 추가한 실험 —
  `unresolved_multihop`(근거 문장 없는 합성 추론) 라벨이 28.8%나 되는 이 데이터셋 특성과
  맞을 수 있다는 가설이지만 단독 검증은 이번이 처음(맨 위 "최종 업데이트" 참고). baseline
  대비 **개선을 확인해야 새 기준으로 승격**.
- 그래프/GAT 없이 순수 baseline만 재현한 결과이므로, 이후 dk_gat 트랙(GAT 증강)과 비교할 때
  baseline 행을 기준으로 삼을 것.
- 추가 실험(하이퍼파라미터 변경)은 이 노트북 셀 4의 커맨드 인자만 바꿔 반복하면 됨 -- 전체
  옵션은 `python -m Scripts.dk_gat.train_gat --help`/`python -m Scripts.dk_gat.train_baseline
  --help` 참고.
